# Notebook 02 — AI Governance Evaluation (G1–G4)
### XAI for FPG Prediction: Algorithmic Fairness in Health Insurance Underwriting
**Paper §3.6, §4.3–4.7**

Four-pillar governance scorecard:

| Pillar | Metric | Key indicator |
|--------|--------|---------------|
| **G1 Fairness** | Δ-RMSE · Δ-StageAccuracy · Prediction Bias | Group-level equity |
| **G2 Robustness** | CV Gap · Temporal Validation · Perturbation | Stability across conditions |
| **G3 Transparency** | SHAP HHI + 95% Bootstrap CI | Variable concentration |
| **G4 Accountability** | Model Card (13 items) | Documentation completeness |

> **Prerequisite**: Run `01_regression_models.ipynb` first.  
> Loads artefacts from `outputs/`.


## 1. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          11,
    "axes.unicode_minus": False,
    "figure.dpi":         150,
})

SEED       = 42
DPI        = 300
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)

# ── Group configuration (must match Notebook 01) ──────────────────────────────
GROUP_CONFIG = {
    "Young_Male":    {"age_group": 0.0, "sex_code": 1.0},
    "Young_Female":  {"age_group": 0.0, "sex_code": 2.0},
    "Middle_Male":   {"age_group": 1.0, "sex_code": 1.0},
    "Middle_Female": {"age_group": 1.0, "sex_code": 2.0},
    "Elderly_Male":  {"age_group": 2.0, "sex_code": 1.0},
    "Elderly_Female":{"age_group": 2.0, "sex_code": 2.0},
}
GROUP_LABELS = {
    "Young_Male":    "Young Male",    "Young_Female":  "Young Female",
    "Middle_Male":   "Middle-aged Male", "Middle_Female": "Middle-aged Female",
    "Elderly_Male":  "Elderly Male",  "Elderly_Female":"Elderly Female",
}
GROUP_COLORS = {
    "Young_Male":    "#1565C0", "Young_Female":  "#90CAF9",
    "Middle_Male":   "#E65100", "Middle_Female": "#FFCC80",
    "Elderly_Male":  "#2E7D32", "Elderly_Female":"#A5D6A7",
}

def assign_glucose_stage(fpg):
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"

log.info("Setup complete.")


2026-04-27 15:11:54,691 | INFO | Setup complete.


## 2. Load Artefacts from Notebook 01

In [2]:
df_final          = pd.read_parquet(OUTPUT_DIR / "df_final.parquet")
all_results       = joblib.load(OUTPUT_DIR / "all_results.pkl")
best_models       = joblib.load(OUTPUT_DIR / "best_models.pkl")
best_algo_name    = joblib.load(OUTPUT_DIR / "best_algo_name.pkl")
best_params_store = joblib.load(OUTPUT_DIR / "best_params_store.pkl")
shap_results      = joblib.load(OUTPUT_DIR / "shap_results.pkl")
cv_df             = joblib.load(OUTPUT_DIR / "cv_df.pkl")
temporal_results  = joblib.load(OUTPUT_DIR / "temporal_results.pkl")
hhi_results       = joblib.load(OUTPUT_DIR / "hhi_results.pkl")

NUM_FEATURES = [
    "BMI", "WaistCirc", "Weight",
    "Energy_kcal", "Carb_g",  "Sugar_g",  "Sodium_mg",
    "Fat_g",       "SatFat_g","Fiber_g",   "Potassium_mg", "Protein_g",
]
CAT_FEATURES = [
    "ObesityStatus",    "WeightChangeStatus", "WeightLossAmount", "WeightGainAmount",
    "DrinkingFrequency","DrinkingAmount",      "SmokingStatus",
    "VigorousAct_Work", "VigorousAct_Leisure","ModerateAct_Work",
    "WalkingActivity",  "AerobicRate",         "BreakfastFreq",
    "StressLevel",      "StressAwareness",
    "IncomeQuartile",   "HouseholdIncome",     "EducationLevel", "HealthScreening",
]
X_FEATURES = NUM_FEATURES + CAT_FEATURES

log.info("Artefacts loaded. df_final: %s", df_final.shape)


2026-04-27 15:11:56,962 | INFO | Artefacts loaded. df_final: (16677, 36)


## 3. Governance Thresholds & Scoring Utilities

In [3]:
# ── Thresholds (documented with literature citations) ────────────────────────
THRESHOLDS = {
    # G1 Fairness
    "G1_delta_rmse":   {"ok": 3.0,  "caution": 6.0,
        "cite": "Obermeyer et al. (2019); group RMSE disparity in medical AI"},
    "G1_stage_acc":    {"ok": 0.10, "caution": 0.20,
        "cite": "Chouldechova (2017); adapted for glucose-stage classification"},
    "G1_bias":         {"ok": 2.0,  "caution": 5.0,
        "cite": "Mean signed prediction error; clinical AI recommendation"},
    # G2 Robustness
    "G2_cv_gap":       {"ok": 1.0,  "caution": 2.0,
        "cite": "Varma & Simon (2006); nested CV stability criterion"},
    "G2_temporal":     {"ok": 1.5,  "caution": 3.5,
        "cite": "Nestor et al. (2019); degradation-only judgement"},
    "G2_perturbation": {"ok": 1.0,  "caution": 2.5,
        "cite": "Ghorbani & Zou (2019); input robustness benchmark"},
    # G3 Transparency
    "G3_hhi":          {"ok": 0.18, "caution": 0.25,
        "cite": "Lundberg et al. (2020); SHAP concentration criterion"},
    # G4 Accountability
    "G4_model_card":   {"ok": 0.90, "caution": 0.70,
        "cite": "Mitchell et al. (2019); Model Cards for Model Reporting"},
}

# Governance weights (G1 highest — fairness failures most consequential)
WEIGHTS = {
    "g1": 0.35, "g2": 0.25, "g3": 0.20, "g4": 0.20,
}

def verdict(value: float, key: str, low_is_good: bool = True) -> str:
    """Return OK / Caution / Risk verdict for a governance metric."""
    t = THRESHOLDS[key]
    if low_is_good:
        if value <= t["ok"]:      return "✓ OK"
        if value <= t["caution"]: return "△ Caution"
        return "✗ Risk"
    else:
        if value >= t["ok"]:      return "✓ OK"
        if value >= t["caution"]: return "△ Caution"
        return "✗ Risk"

def normalise(value: float, ok_val: float, risk_val: float,
              low_is_good: bool = True, floor: float = 0.05) -> float:
    """Map raw metric value to [floor, 1.0] normalised score."""
    if low_is_good:
        if value <= ok_val:   return 1.0
        if value >= risk_val: return floor
        return floor + (1.0 - floor) * (1.0 - (value - ok_val) / (risk_val - ok_val))
    else:
        if value >= ok_val:   return 1.0
        if value <= risk_val: return floor
        return floor + (1.0 - floor) * ((value - risk_val) / (ok_val - risk_val))

log.info("Thresholds and utilities defined.")


2026-04-27 15:11:56,996 | INFO | Thresholds and utilities defined.


## 4. G1 — Fairness

In [4]:
# ── Per-group fairness metrics ───────────────────────────────────────────────
g1_records = []

for grp, cfg in GROUP_CONFIG.items():
    df_g  = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy()
    model = best_models.get(grp)
    algo  = best_algo_name.get(grp)
    if model is None:
        continue

    X     = df_g[X_FEATURES]
    y     = df_g["FPG"]
    _, X_val, _, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED)

    y_pred = model.predict(X_val)
    rmse   = float(np.sqrt(mean_squared_error(y_val, y_pred)))
    mae    = float(mean_absolute_error(y_val, y_pred))
    r2     = float(r2_score(y_val, y_pred))
    bias   = float(np.mean(y_pred - y_val))   # positive = over-prediction

    # Glucose-stage classification accuracy
    stage_true = pd.Series(y_val.values).apply(assign_glucose_stage)
    stage_pred = pd.Series(y_pred).apply(assign_glucose_stage)
    stage_acc  = float((stage_true == stage_pred).mean())

    g1_records.append({
        "Group":       GROUP_LABELS[grp],
        "Algorithm":   algo,
        "RMSE":        round(rmse, 4),
        "MAE":         round(mae,  4),
        "R2":          round(r2,   4),
        "Bias":        round(bias, 4),
        "Stage_Acc":   round(stage_acc, 4),
        "N_holdout":   len(y_val),
        "_grp_key":    grp,
    })

g1_df = pd.DataFrame(g1_records)

# ── G1 summary deltas ─────────────────────────────────────────────────────────
delta_rmse      = round(g1_df["RMSE"].max()     - g1_df["RMSE"].min(), 4)
delta_stage_acc = round(g1_df["Stage_Acc"].max()- g1_df["Stage_Acc"].min(), 4)
delta_bias      = round(g1_df["Bias"].abs().max()- g1_df["Bias"].abs().min(), 4)

g1_summary = pd.DataFrame([{
    "Metric":  "Δ-RMSE (max − min)",
    "Value":   delta_rmse,
    "Verdict": verdict(delta_rmse, "G1_delta_rmse"),
    "Reference": THRESHOLDS["G1_delta_rmse"]["cite"],
},{
    "Metric":  "Δ-StageAccuracy (max − min)",
    "Value":   delta_stage_acc,
    "Verdict": verdict(delta_stage_acc, "G1_stage_acc"),
    "Reference": THRESHOLDS["G1_stage_acc"]["cite"],
},{
    "Metric":  "Δ-Bias (max |bias| − min |bias|)",
    "Value":   delta_bias,
    "Verdict": verdict(delta_bias, "G1_bias"),
    "Reference": THRESHOLDS["G1_bias"]["cite"],
}])

print("── G1 Per-Group Metrics ──")
print(g1_df.drop(columns=["_grp_key"]).to_string(index=False))
print("\n── G1 Fairness Summary ──")
print(g1_summary.to_string(index=False))

g1_df.drop(columns=["_grp_key"]).to_csv(
    OUTPUT_DIR / "table_g1_fairness.csv", index=False, encoding="utf-8")


── G1 Per-Group Metrics ──
             Group Algorithm    RMSE     MAE     R2    Bias  Stage_Acc  N_holdout
        Young Male      LGBM 18.3860  8.5718 0.0728 -0.8545     0.7708        349
      Young Female      LGBM 15.8624  7.1404 0.0813 -0.4536     0.9108        415
  Middle-aged Male      LGBM 25.9668 15.2688 0.0315 -1.0738     0.4520        646
Middle-aged Female       XGB 17.3049 10.5567 0.1536  0.6881     0.6291        887
      Elderly Male      LGBM 21.4421 15.7160 0.0507  0.5625     0.4484        455
    Elderly Female        RF 20.1553 13.5241 0.0698  0.5230     0.4376        585

── G1 Fairness Summary ──
                          Metric   Value Verdict                                                     Reference
              Δ-RMSE (max − min) 10.1044  ✗ Risk   Obermeyer et al. (2019); group RMSE disparity in medical AI
     Δ-StageAccuracy (max − min)  0.4732  ✗ Risk Chouldechova (2017); adapted for glucose-stage classification
Δ-Bias (max |bias| − min |bias|)  0.620

## 5. G2 — Robustness

In [5]:
# ── G2-a: CV stability ───────────────────────────────────────────────────────
g2a_records = []
for _, row in cv_df.iterrows():
    ho_rmse = float(row["HO_RMSE"])
    cv_mean = float(row["CV_RMSE_mean"])
    gap     = round(abs(cv_mean - ho_rmse), 4)
    g2a_records.append({
        "Group":         row["Group"],
        "Algorithm":     row["Best_Algorithm"],
        "HO_RMSE":       ho_rmse,
        "CV_RMSE_mean":  cv_mean,
        "CV_RMSE_std":   float(row["CV_RMSE_std"]),
        "Gap":           gap,
        "G2a_verdict":   verdict(gap, "G2_cv_gap"),
        "Reference":     THRESHOLDS["G2_cv_gap"]["cite"],
    })
g2a_df = pd.DataFrame(g2a_records)
print("── G2-a: CV Stability ──")
print(g2a_df.to_string(index=False))

# ── G2-b: Temporal validation (loaded from NB01) ──────────────────────────────
print("\n── G2-b: Temporal Validation ──")
g2b_records = []
for grp, res in temporal_results.items():
    if res is None:
        continue
    g2b_records.append({
        "Group":          GROUP_LABELS[grp],
        "Algorithm":      best_algo_name[grp],
        "HO_RMSE":        res["HO_RMSE"],
        "Temporal_RMSE":  res["Temporal_RMSE"],
        "Delta":          res["Delta"],
        "Direction":      res["Direction"],
        "G2b_verdict":    res["G2b_Verdict"],
        "Reference":      THRESHOLDS["G2_temporal"]["cite"],
    })
g2b_df = pd.DataFrame(g2b_records)
print(g2b_df.to_string(index=False))

# ── G2-c: Feature perturbation (±10% Gaussian noise, 5 repeats) ──────────────
print("\n── G2-c: Feature Perturbation (±10% Gaussian noise, 5 repeats) ──")
g2c_records = []
rng = np.random.RandomState(SEED)

for grp, cfg in GROUP_CONFIG.items():
    df_g  = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy()
    model = best_models.get(grp)
    if model is None:
        continue
    _, X_val, _, y_val = train_test_split(
        df_g[X_FEATURES], df_g["FPG"], test_size=0.20, random_state=SEED)
    rmse_base = float(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))

    drops = []
    for _ in range(5):
        X_noisy = X_val.copy()
        for col in X_noisy.columns:
            std = X_noisy[col].std()
            if std > 0:
                X_noisy[col] += rng.normal(0, std * 0.10, len(X_noisy))
        drops.append(abs(float(np.sqrt(mean_squared_error(
            y_val, model.predict(X_noisy)))) - rmse_base))

    mean_drop = round(np.mean(drops), 4)
    g2c_records.append({
        "Group":         GROUP_LABELS[grp],
        "Algorithm":     best_algo_name[grp],
        "RMSE_base":     round(rmse_base, 4),
        "RMSE_drop_mean":mean_drop,
        "RMSE_drop_std": round(np.std(drops), 4),
        "G2c_verdict":   verdict(mean_drop, "G2_perturbation"),
        "Reference":     "±10% Gaussian noise, 5 repeats",
    })
g2c_df = pd.DataFrame(g2c_records)
print(g2c_df.to_string(index=False))

# Save
pd.concat([
    g2a_df.assign(dimension="G2a_CV_Stability"),
    g2b_df.assign(dimension="G2b_Temporal"),
    g2c_df.assign(dimension="G2c_Perturbation"),
], ignore_index=True).to_csv(
    OUTPUT_DIR / "table_g2_robustness.csv", index=False, encoding="utf-8")


── G2-a: CV Stability ──
             Group Algorithm  HO_RMSE  CV_RMSE_mean  CV_RMSE_std    Gap G2a_verdict                                           Reference
        Young Male      LGBM  18.3860       15.1107       3.9770 3.2753      ✗ Risk Varma & Simon (2006); nested CV stability criterion
      Young Female      LGBM  15.8624       12.8839       2.4338 2.9785      ✗ Risk Varma & Simon (2006); nested CV stability criterion
  Middle-aged Male      LGBM  25.9668       25.1956       2.0116 0.7712        ✓ OK Varma & Simon (2006); nested CV stability criterion
Middle-aged Female       XGB  17.3049       19.7553       1.7777 2.4504      ✗ Risk Varma & Simon (2006); nested CV stability criterion
      Elderly Male      LGBM  21.4421       25.5796       3.2447 4.1375      ✗ Risk Varma & Simon (2006); nested CV stability criterion
    Elderly Female        RF  20.1553       21.8464       1.0217 1.6911   △ Caution Varma & Simon (2006); nested CV stability criterion

── G2-b: Temporal Vali

## 6. G3 — Transparency (SHAP HHI)

In [6]:
# HHI results were computed in Notebook 01
# Here we format for the paper and add verdict
g3_records = []
for grp, hhi in hhi_results.items():
    g3_records.append({
        "Group":         GROUP_LABELS[grp],
        "Algorithm":     best_algo_name[grp],
        "SHAP_HHI":      hhi["HHI"],
        "CI_95_lower":   hhi["CI_lo"],
        "CI_95_upper":   hhi["CI_hi"],
        "G3_verdict":    verdict(hhi["HHI"], "G3_hhi"),
        "Interpretation":
            ("High concentration — single-variable dominance" if hhi["HHI"] > 0.25
             else "Moderate concentration" if hhi["HHI"] > 0.18
             else "Well-distributed importance"),
        "Reference":     THRESHOLDS["G3_hhi"]["cite"],
    })

g3_df = pd.DataFrame(g3_records)
print("── G3: Transparency (SHAP HHI with 95% Bootstrap CI) ──")
print(g3_df.to_string(index=False))
g3_df.to_csv(OUTPUT_DIR / "table_g3_transparency.csv", index=False, encoding="utf-8")


── G3: Transparency (SHAP HHI with 95% Bootstrap CI) ──
             Group Algorithm  SHAP_HHI  CI_95_lower  CI_95_upper G3_verdict              Interpretation                                            Reference
        Young Male      LGBM    0.1389       0.1323       0.1463       ✓ OK Well-distributed importance Lundberg et al. (2020); SHAP concentration criterion
      Young Female      LGBM    0.1515       0.1382       0.1664       ✓ OK Well-distributed importance Lundberg et al. (2020); SHAP concentration criterion
  Middle-aged Male      LGBM    0.0742       0.0716       0.0772       ✓ OK Well-distributed importance Lundberg et al. (2020); SHAP concentration criterion
Middle-aged Female       XGB    0.0930       0.0886       0.0978       ✓ OK Well-distributed importance Lundberg et al. (2020); SHAP concentration criterion
      Elderly Male      LGBM    0.0940       0.0879       0.1015       ✓ OK Well-distributed importance Lundberg et al. (2020); SHAP concentration criterion
  

## 7. G4 — Accountability (Model Card)

In [7]:
GITHUB_URL = ""   # set to your repo URL when available

MODEL_CARD = {
    # Documentation
    "Intended purpose documented":                          True,
    "Training data source and period stated":               True,
    "Medicated-diabetic exclusion criterion stated":        True,
    "Performance metrics reported (RMSE/MAE/R²)":          True,
    "Limitations and failure modes disclosed":              True,
    # Fairness
    "Disaggregated performance by group":                   True,
    # Explainability
    "SHAP Waterfall plots available":                       True,
    "DiCE counterfactual paths provided":                   True,
    "LLM personalised health reports generated":            True,
    # Governance gaps (to be resolved before deployment)
    "Code publicly available (GitHub)":                     bool(GITHUB_URL),
    "Model versioning and update plan":                     True,
    "Human-in-the-loop procedure defined":                  False,   # ← gap
    "Privacy Impact Assessment (PIA) conducted":            False,   # ← gap
}

total     = len(MODEL_CARD)
fulfilled = sum(MODEL_CARD.values())
mc_score  = round(fulfilled / total, 4)
mc_verdict= verdict(mc_score, "G4_model_card", low_is_good=False)

print(f"Model Card Score: {mc_score:.2f}  ({fulfilled}/{total} items)  [{mc_verdict}]\n")
print("── Item Details ──")
for item, status in MODEL_CARD.items():
    icon = "✓" if status else "✗"
    print(f"  [{icon}] {item}")

missing = [k for k, v in MODEL_CARD.items() if not v]
if missing:
    print(f"\n  ⚠ Unresolved gaps ({len(missing)}):")
    for m in missing:
        print(f"    - {m}")

g4_df = pd.DataFrame([
    {"Group": GROUP_LABELS[g], "MC_Score": mc_score,
     "G4_verdict": mc_verdict, "Fulfilled": f"{fulfilled}/{total}"}
    for g in GROUP_CONFIG
])
g4_df.to_csv(OUTPUT_DIR / "table_g4_accountability.csv", index=False, encoding="utf-8")


Model Card Score: 0.77  (10/13 items)  [△ Caution]

── Item Details ──
  [✓] Intended purpose documented
  [✓] Training data source and period stated
  [✓] Medicated-diabetic exclusion criterion stated
  [✓] Performance metrics reported (RMSE/MAE/R²)
  [✓] Limitations and failure modes disclosed
  [✓] Disaggregated performance by group
  [✓] SHAP Waterfall plots available
  [✓] DiCE counterfactual paths provided
  [✓] LLM personalised health reports generated
  [✗] Code publicly available (GitHub)
  [✓] Model versioning and update plan
  [✗] Human-in-the-loop procedure defined
  [✗] Privacy Impact Assessment (PIA) conducted

  ⚠ Unresolved gaps (3):
    - Code publicly available (GitHub)
    - Human-in-the-loop procedure defined
    - Privacy Impact Assessment (PIA) conducted


## 8. Governance Composite Score

In [8]:
# ── Normalise each pillar to [0, 1] ─────────────────────────────────────────
s_g1_rmse  = normalise(delta_rmse,      ok_val=3.0,  risk_val=6.0)
s_g1_stage = normalise(delta_stage_acc, ok_val=0.10, risk_val=0.20)
s_g1       = round((s_g1_rmse + s_g1_stage) / 2, 4)

s_g2a = normalise(g2a_df["Gap"].mean(),        ok_val=1.0,  risk_val=2.0)
s_g2c = normalise(g2c_df["RMSE_drop_mean"].mean(), ok_val=1.0, risk_val=2.5)
# Temporal: only degradation counts as risk
if len(g2b_df) > 0:
    max_deg = g2b_df.loc[g2b_df["Delta"] > 0, "Delta"]
    s_g2b   = normalise(max_deg.max() if len(max_deg) else 0.0, ok_val=1.5, risk_val=3.5)
else:
    s_g2b = 1.0
s_g2 = round((s_g2a + s_g2b + s_g2c) / 3, 4)

s_g3 = normalise(g3_df["SHAP_HHI"].mean(), ok_val=0.18, risk_val=0.25)

s_g4 = normalise(mc_score, ok_val=0.90, risk_val=0.70, low_is_good=False)

composite = round(
    WEIGHTS["g1"] * s_g1 +
    WEIGHTS["g2"] * s_g2 +
    WEIGHTS["g3"] * s_g3 +
    WEIGHTS["g4"] * s_g4, 4
)

overall_verdict = ("✓ Green — Deploy" if composite >= 0.80
                   else "△ Amber — Conditional deploy" if composite >= 0.60
                   else "✗ Red — Do not deploy")

scorecard = pd.DataFrame([
    {"Pillar":"G1 Fairness",      "Weight":WEIGHTS["g1"], "Score":s_g1,
     "Verdict": verdict(delta_rmse,"G1_delta_rmse")},
    {"Pillar":"G2 Robustness",    "Weight":WEIGHTS["g2"], "Score":s_g2,
     "Verdict": "Caution" if s_g2 < 0.8 else "OK"},
    {"Pillar":"G3 Transparency",  "Weight":WEIGHTS["g3"], "Score":s_g3,
     "Verdict": g3_df["G3_verdict"].mode()[0]},
    {"Pillar":"G4 Accountability","Weight":WEIGHTS["g4"], "Score":s_g4,
     "Verdict": mc_verdict},
    {"Pillar":"COMPOSITE",        "Weight":1.00, "Score":composite,
     "Verdict": overall_verdict},
])

print(scorecard.to_string(index=False))
scorecard.to_csv(OUTPUT_DIR / "table_governance_scorecard.csv",
                 index=False, encoding="utf-8")
log.info("Composite score: %.4f  [%s]", composite, overall_verdict)


           Pillar  Weight  Score               Verdict
      G1 Fairness    0.35 0.0500                ✗ Risk
    G2 Robustness    0.25 0.4561               Caution
  G3 Transparency    0.20 1.0000                  ✓ OK
G4 Accountability    0.20 0.3787             △ Caution
        COMPOSITE    1.00 0.4073 ✗ Red — Do not deploy


2026-04-27 15:11:59,422 | INFO | Composite score: 0.4073  [✗ Red — Do not deploy]


## 9. Figures

In [9]:
# ── Figure: G1 Fairness — RMSE and Stage Accuracy ───────────────────────────
labels = [GROUP_LABELS[g] for g in GROUP_CONFIG]
colors = [GROUP_COLORS[g] for g in GROUP_CONFIG]
rmse_vals  = [g1_df.loc[g1_df["Group"]==GROUP_LABELS[g], "RMSE"].values[0]
              for g in GROUP_CONFIG]
stage_vals = [g1_df.loc[g1_df["Group"]==GROUP_LABELS[g], "Stage_Acc"].values[0]
              for g in GROUP_CONFIG]
bias_vals  = [g1_df.loc[g1_df["Group"]==GROUP_LABELS[g], "Bias"].values[0]
              for g in GROUP_CONFIG]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Left: RMSE ────────────────────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(labels, rmse_vals, color=colors, alpha=0.85,
              edgecolor="white", linewidth=0.6)
for bar, val in zip(bars, rmse_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.15, f"{val:.2f}",
            ha="center", va="bottom", fontsize=8.5)
ax.axhline(np.mean(rmse_vals), color="#C62828", linestyle="--",
           lw=1.5, alpha=0.8, label=f"Mean RMSE = {np.mean(rmse_vals):.2f}")
# Δ annotation
ax.annotate("",
    xy=(5.45, max(rmse_vals)), xytext=(5.45, min(rmse_vals)),
    arrowprops=dict(arrowstyle="<->", color="#C62828", lw=1.5))
ax.text(5.55, (max(rmse_vals)+min(rmse_vals))/2,
        f"Δ = {delta_rmse:.2f}\n{verdict(delta_rmse,'G1_delta_rmse')}",
        fontsize=8, color="#C62828", va="center")
ax.set_ylabel("RMSE (mg/dL)", fontsize=10)
ax.set_title("G1(a): RMSE Fairness", fontsize=11, fontweight="bold")
ax.tick_params(axis="x", rotation=30, labelsize=8)
ax.legend(fontsize=8)
ax.spines[["top","right"]].set_visible(False)

# ── Middle: Stage Accuracy ────────────────────────────────────────────────────
ax = axes[1]
bar_colors_sa = ["#C62828" if v < 0.50 else "#F9A825" if v < 0.75
                 else "#2E7D32" for v in stage_vals]
bars = ax.bar(labels, stage_vals, color=bar_colors_sa, alpha=0.85,
              edgecolor="white", linewidth=0.6)
for bar, val in zip(bars, stage_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.008, f"{val:.3f}",
            ha="center", va="bottom", fontsize=8.5)
ax.axhline(0.80, color="#2E7D32", linestyle="--", lw=1.2, alpha=0.7,
           label="Acceptable threshold (0.80)")
ax.set_ylabel("Stage Accuracy", fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_title("G1(b): Glucose-Stage Classification Accuracy", fontsize=11, fontweight="bold")
ax.tick_params(axis="x", rotation=30, labelsize=8)
legend_patches = [
    mpatches.Patch(color="#2E7D32", label="≥ 0.75 (OK)"),
    mpatches.Patch(color="#F9A825", label="0.50–0.75 (Caution)"),
    mpatches.Patch(color="#C62828", label="< 0.50 (Risk)"),
]
ax.legend(handles=legend_patches, fontsize=8)
ax.spines[["top","right"]].set_visible(False)

# ── Right: Prediction Bias ────────────────────────────────────────────────────
ax = axes[2]
bias_colors = ["#E57373" if v > 0 else "#64B5F6" for v in bias_vals]
bars = ax.bar(labels, bias_vals, color=bias_colors, alpha=0.85,
              edgecolor="white", linewidth=0.6)
for bar, val in zip(bars, bias_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.15 if val >= 0 else -0.35),
            f"{val:+.2f}", ha="center", va="bottom", fontsize=8.5)
ax.axhline(0,   color="black",  lw=0.8)
ax.axhline(2,   color="#F9A825", linestyle="--", lw=1.2, alpha=0.7)
ax.axhline(-2,  color="#F9A825", linestyle="--", lw=1.2, alpha=0.7,
           label="Caution band (±2 mg/dL)")
ax.set_ylabel("Prediction Bias (mg/dL)\n(positive = over-prediction)", fontsize=10)
ax.set_title("G1(c): Prediction Bias", fontsize=11, fontweight="bold")
ax.tick_params(axis="x", rotation=30, labelsize=8)
ax.legend(fontsize=8)
ax.spines[["top","right"]].set_visible(False)

plt.suptitle("G1 Fairness Evaluation — Predictive Equity across Demographic Groups",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_g1_fairness.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("G1 figure saved.")


2026-04-27 15:12:00,998 | INFO | G1 figure saved.


In [10]:
# ── Governance radar chart (G1–G4) ──────────────────────────────────────────
categories = ["G1\nFairness", "G2\nRobustness",
              "G3\nTransparency", "G4\nAccountability"]
scores_raw = [s_g1, s_g2, s_g3, s_g4]
N          = len(categories)
angles     = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles    += angles[:1]
scores_cl  = scores_raw + [scores_raw[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles, scores_cl, "o-", color="#1565C0", lw=2.5, ms=8)
ax.fill(angles, scores_cl, color="#1565C0", alpha=0.20)

# Reference rings
for level, ls, col, label in [
    (0.80, "--", "#2E7D32", "Deploy threshold (0.80)"),
    (0.60, ":",  "#F9A825", "Conditional threshold (0.60)"),
]:
    ax.plot(angles, [level]*(N+1), ls=ls, color=col, lw=1.2, alpha=0.65, label=label)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11, fontweight="bold")
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], fontsize=8, color="gray")

# Score labels on spokes
for angle, score in zip(angles[:-1], scores_raw):
    ax.text(angle, score + 0.09, f"{score:.2f}",
            ha="center", va="center", fontsize=10, fontweight="bold",
            color="#1565C0",
            bbox=dict(boxstyle="round,pad=0.2", fc="white",
                      ec="#1565C0", alpha=0.85))

grade = ("🟢 Green" if composite >= 0.80
         else "🟡 Amber" if composite >= 0.60 else "🔴 Red")
ax.set_title(
    f"AI Governance Scorecard — FPG Prediction Pipeline\n"
    f"Composite Score: {composite:.2f}   [{grade}]",
    fontsize=12, fontweight="bold", pad=28,
)
ax.legend(fontsize=9, loc="lower right", bbox_to_anchor=(1.25, -0.08))

plt.tight_layout(pad=2.5)
fig.savefig(OUTPUT_DIR / "fig_governance_radar.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("Governance radar saved. Composite = %.4f", composite)


2026-04-27 15:12:01,566 | INFO | Governance radar saved. Composite = 0.4073


In [11]:
# ── Algorithm × Group RMSE heatmap (full 6×6) ──────────────────────────────
ALGORITHMS = ["LR", "Ridge", "RF", "LGBM", "XGB", "MLP"]
rmse_mat = pd.DataFrame(index=list(GROUP_CONFIG.keys()), columns=ALGORITHMS)
for grp in GROUP_CONFIG:
    for algo in ALGORITHMS:
        m = all_results.get(grp, {}).get(algo)
        rmse_mat.loc[grp, algo] = m["RMSE"] if m else np.nan
rmse_mat.index = [GROUP_LABELS[g] for g in rmse_mat.index]
rmse_mat = rmse_mat.astype(float)

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(rmse_mat, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "RMSE (mg/dL)", "shrink": 0.8},
            ax=ax)
ax.set_title(
    "Hold-Out RMSE: All Algorithms × All Demographic Groups\n"
    "(★ = best per group; shading: darker = higher RMSE)",
    fontsize=12, fontweight="bold", pad=10)
ax.set_xlabel("Algorithm", fontsize=11)
ax.set_ylabel("Demographic Group", fontsize=11)

# Star best model per group
for i, grp in enumerate(GROUP_CONFIG):
    best_a = best_algo_name.get(grp)
    if best_a and best_a in ALGORITHMS:
        j = ALGORITHMS.index(best_a)
        ax.text(j + 0.5, i + 0.18, "★", ha="center", va="center",
                color="navy", fontsize=16, fontweight="bold")

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_rmse_heatmap_full.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("RMSE heatmap saved.")


2026-04-27 15:12:02,264 | INFO | RMSE heatmap saved.


## 10. Save All Governance Outputs

In [12]:
# Consolidated governance summary
gov_summary = g1_df.drop(columns=["_grp_key"]).copy()
gov_summary = gov_summary.merge(
    g2a_df[["Group","Gap","G2a_verdict"]].rename(
        columns={"Gap":"CV_Gap","G2a_verdict":"G2a"}),
    on="Group", how="left")
gov_summary = gov_summary.merge(
    g2c_df[["Group","RMSE_drop_mean","G2c_verdict"]].rename(
        columns={"RMSE_drop_mean":"Perturb_drop","G2c_verdict":"G2c"}),
    on="Group", how="left")
gov_summary = gov_summary.merge(
    g3_df[["Group","SHAP_HHI","CI_95_lower","CI_95_upper","G3_verdict"]],
    on="Group", how="left")
gov_summary["MC_Score"]   = mc_score
gov_summary["G4_verdict"] = mc_verdict

gov_summary.to_csv(OUTPUT_DIR / "table_governance_full.csv",
                   index=False, encoding="utf-8")

log.info("=" * 55)
log.info("Notebook 02 complete. Outputs:")
for f in sorted(OUTPUT_DIR.glob("table_g*.csv")):
    log.info("  %s", f.name)
for f in sorted(OUTPUT_DIR.glob("fig_g*.png")):
    log.info("  %s", f.name)
log.info("  fig_governance_radar.png")
log.info("  fig_rmse_heatmap_full.png")
log.info("=" * 55)


2026-04-27 15:12:02,322 | INFO | =======================================================
2026-04-27 15:12:02,324 | INFO | Notebook 02 complete. Outputs:
2026-04-27 15:12:02,330 | INFO |   table_g1_fairness.csv
2026-04-27 15:12:02,331 | INFO |   table_g2_robustness.csv
2026-04-27 15:12:02,331 | INFO |   table_g3_transparency.csv
2026-04-27 15:12:02,332 | INFO |   table_g4_accountability.csv
2026-04-27 15:12:02,334 | INFO |   table_governance_full.csv
2026-04-27 15:12:02,335 | INFO |   table_governance_scorecard.csv
2026-04-27 15:12:02,337 | INFO |   fig_g1_fairness.png
2026-04-27 15:12:02,338 | INFO |   fig_governance_radar.png
2026-04-27 15:12:02,341 | INFO |   fig_governance_radar.png
2026-04-27 15:12:02,347 | INFO |   fig_rmse_heatmap_full.png
2026-04-27 15:12:02,349 | INFO | =======================================================
